In [143]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [144]:
df = yf.download("META", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2016-10-07,127.766304,128.023832,127.112562,127.815818
1,2016-10-10,129.004456,129.460083,127.974313,128.449755
2,2016-10-11,127.657341,129.400639,127.013498,128.994525
3,2016-10-12,127.825729,128.429943,127.241330,127.786100
4,2016-10-13,126.607391,127.033312,125.547543,126.993698
...,...,...,...,...,...
2507,2026-09-30,725.179993,738.250000,721.299988,730.270020
2508,2026-10-01,725.929993,735.880005,721.510010,728.530029
2509,2026-10-02,728.080017,741.580017,726.200012,733.250000
2510,2026-10-05,741.900024,746.700012,727.119995,727.710022


In [145]:
def SMI(data, n1, n2, n3, n4):
    hh = data.High.rolling(n1).max()
    ll = data.Low.rolling(n1).min()
    mid = hh - ll
    med = (hh + ll) / 2
    k1 = (data['Close'] - med)
    d1 = k1.ewm(span=n2).mean()
    d = d1.ewm(span=n2).mean()
    k = mid.ewm(span=n2).mean()
    smi = (d / k) * 100
    sig = smi.rolling(n3).mean()
    histo1 = smi - sig
    histo = histo1.ewm(span=n4).mean()
    return smi, sig, histo

df['SMI'], df['SIG'], df['HISTO'] = SMI(df, 10, 3, 5, 9)
df.dropna(inplace=True)
    

In [146]:
def signal(data):
    signal = [0] * len(df)
    for i in range(0,len(df)):
        if (data.HISTO.iloc[i] < 0) &\
        (data.HISTO.iloc[i] > data.HISTO.iloc[i-1]):
            signal[i] = 1
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)

def entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    elif x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['entries'] = df.apply(lambda x: entries(x), axis=1)
print(df['signal'].value_counts())
df.shape

signal
0    1921
1     578
Name: count, dtype: int64


(2499, 10)

In [147]:
df.set_index('Date', inplace=True)

In [148]:
bar = 2200
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'META'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['entries'], mode="markers",
                marker=dict(size=7, color="White"),
                name="Trade Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.SMI, 
                         line=dict(color='red', width=1)),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.SIG, 
                         line=dict(color='white', width=1)),
                         row=2, col=1)

fig.add_trace(go.Bar(x=df1.index, 
                         y=df1.HISTO, 
                         marker=dict(color='gray')),
                         row=2, col=1)

fig.update_layout(
    annotations=[dict(text="META - Stochastic Momentum Oscillator Strategy",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=0.02,
                    y=1.1,
                    showarrow=False),
                dict(text="SMI",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=0.02,
                    y=0.3,
                    showarrow=False)])

fig.update_layout(autosize=False, width=1100, height=720, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.show()

In [162]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, sl=0.9*price, tp=1.045*price)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Backtest.run:   0%|          | 0/2498 [00:00<?, ?bar/s]

Start                     2016-10-26 00:00:00
End                       2026-10-06 00:00:00
Duration                   3632 days 00:00:00
Exposure Time [%]                    54.14166
Equity Final [$]                 520064.49112
Equity Peak [$]                  520064.49112
Commissions [$]                   31335.36972
Return [%]                          420.06449
Buy & Hold Return [%]                469.2589
Return (Ann.) [%]                    18.08846
Volatility (Ann.) [%]                29.35854
CAGR [%]                             12.11982
Sharpe Ratio                          0.61612
Sortino Ratio                         1.07234
Calmar Ratio                          0.66424
Alpha [%]                           247.85889
Beta                                  0.36697
Max. Drawdown [%]                   -27.23195
Avg. Drawdown [%]                     -4.8614
Max. Drawdown Duration      549 days 00:00:00
Avg. Drawdown Duration       47 days 00:00:00
# Trades                          

In [150]:
results = []

if stats is not None:
            results.append({
                '# Trades': stats['# Trades'],
                'Exposure Time [%]': stats['Exposure Time [%]'],
                'Win Rate [%]': stats['Win Rate [%]'],
                'Return [%]': stats['Return [%]'],
                'Buy & Hold Return [%]': stats['Buy & Hold Return [%]'],
                'Max. Drawdown [%]': stats['Max. Drawdown [%]'],
                'Avg. Drawdown [%]': stats['Avg. Drawdown [%]'],
                'Sharpe Ratio': stats['Sharpe Ratio'],
                'Sortino Ratio': stats['Sortino Ratio'],
                'Calmar Ratio': stats['Calmar Ratio'],
                'Best Trade [%]': stats['Best Trade [%]'],
                'Worst Trade [%]': stats['Worst Trade [%]'],
                'Avg. Trade [%]': stats['Avg. Trade [%]'],
                'Profit Factor': stats['Profit Factor'],
                'Expectancy [%]': stats['Expectancy [%]']
            })

results_df = pd.DataFrame(results)
results_df = results_df.T
results_df.columns = ['Value']
results_df.to_csv('stats.xlsx', index=True)

In [151]:
trades = stats['_trades']

def randomised_trades(trades):
    cumulative_return = [0]

    for pct in trades['ReturnPct']:
        cumulative_return.append(cumulative_return[-1] + (pct * 100))

    return cumulative_return

simulations = 100
curves = []

for i in range(simulations):
    new_trades = trades.sample(frac=1).reset_index(drop=True)
    equity_curve = randomised_trades(new_trades)
    curves.append(equity_curve)

mc = go.Figure()

for equity_curve in curves:
    mc.add_trace(go.Scatter(y=equity_curve, mode='lines', opacity=0.6, showlegend=False))

mc.update_layout(
    title='META SMI Strategy Monte Carlo Simulation',
    xaxis_title='Trade Number',
    yaxis_title='% / Trade',
    template="plotly_dark",
    autosize=False,
    width=1100,
    height=700,
)

mc.update_yaxes(gridcolor="#171717")
mc.update_xaxes(gridcolor="#171717")

mc.show()

In [152]:
trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='META SMI Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.show()

In [153]:
# ---------- Theme ----------
SITE_BG   = "#12181B"
CARD_BG   = "#161D21"
ROW_A     = "#141A1E"
ROW_B     = "#101519"
BORDER    = "rgba(255,255,255,0.08)"
TEXT      = "#E6ECEF"
GREEN     = "#00DF9A"

# Safer for static image export. Use Atkinson only if installed for Kaleido.
FONT_FAM  = "Arial, Helvetica, sans-serif"

# ---------- Load CSV ----------
df2 = pd.read_csv(
    "/Users/benlockhart/Downloads/Blog Strats/1. Published/CMB/stats1.csv",
    header=None,
    names=["Metric", "Value"]
)

df2 = df2.dropna(how="all").copy()

metrics = df2["Metric"].astype(str)
values  = df2["Value"].astype(str)

n = len(df2)

# ---------- Sizing ----------
WIDTH = 500
ROW_H = 34
TOP_MARGIN = 105
BOTTOM_MARGIN = 35
TABLE_H = n * ROW_H
HEIGHT = TOP_MARGIN + TABLE_H + BOTTOM_MARGIN

stripe = [ROW_A if i % 2 == 0 else ROW_B for i in range(n)]
fill = [stripe, stripe]

# ---------- Table ----------
fig = go.Figure(data=[
    go.Table(
        columnwidth=[1.15, 1],
        domain=dict(
            x=[0, 1],
            y=[0, 1]
        ),

        # Hide Plotly's default header row.
        header=dict(
            values=["", ""],
            height=1,
            fill_color=CARD_BG,
            line_color=CARD_BG,
            font=dict(size=1, color=CARD_BG, family=FONT_FAM),
        ),

        cells=dict(
            values=[metrics, values],
            align=["left", "right"],
            height=ROW_H,
            font=dict(size=13, color=TEXT, family=FONT_FAM),
            fill_color=fill,
            line_color=BORDER,
            line_width=1,
        )
    )
])

# ---------- Layout ----------
fig.update_layout(
    width=WIDTH,
    height=HEIGHT,
    paper_bgcolor=SITE_BG,
    plot_bgcolor=CARD_BG,
    margin=dict(l=30, r=30, t=TOP_MARGIN, b=BOTTOM_MARGIN),
)

# ---------- Title ----------
fig.add_annotation(
    text="<b>META - Stochastic Momentum Oscillator Strategy</b>",
    x=0,
    y=1.105,
    xref="paper",
    yref="paper",
    xanchor="left",
    showarrow=False,
    font=dict(size=12, color=TEXT, family=FONT_FAM),
)

# ---------- Brand ----------
fig.add_annotation(
    text=f"<b><span style='color:{GREEN}'>BUBBLE</span><span style='color:white'>ANALYTICS</span></b>",
    x=1,
    y=1.105,
    xref="paper",
    yref="paper",
    xanchor="right",
    showarrow=False,
    font=dict(size=12, family=FONT_FAM),
)

# ---------- Card Border ----------
fig.add_shape(
    type="rect",
    xref="paper",
    yref="paper",
    x0=0,
    y0=0,
    x1=1,
    y1=1,
    line=dict(color=BORDER, width=1),
    fillcolor=CARD_BG,
    layer="below",
)

fig.show()